In [ ]:
!pip install -q sentence-transformers pandas numpy

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("Carregando o modelo all-MiniLM-L6-v2...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")



dados_experimento = [
    {
        "pergunta": "Quais são as orientações antes da internação para a cirurgia cardíaca? Preciso parar de fumar e beber antes da internação para a cirurgia?",
        "resposta_esperada": "Antes da internação para a cirurgia cardíaca, o paciente deve seguir orientações rigorosas de preparo. É altamente recomendável cessar o tabagismo e o consumo de bebidas alcoólicas com antecedência para reduzir riscos cirúrgicos e respiratórios. Além disso, o paciente deve estar ciente de que acordará na UTI sob efeito de anestesia e com dispositivos de monitoramento, sendo extubado poucas horas após o procedimento.",
        "kb1_artigos": "Baseado no contexto fornecido, não há informações específicas sobre orientações antes da internação para a cirurgia cardíaca...",
        "kb2_faq": "Baseado no contexto fornecido, as orientações antes da internação incluem: o paciente irá para a UTI ainda dormindo sob efeito anestésico; o despertador do paciente ocorre poucas horas após a cirurgia...",
        "kb3_combinada": "Sim, é recomendável parar de fumar e beber antes da internação...",
    },
    {
        "pergunta": "Existe algum exame específico que preciso fazer antes da internação?",
        "resposta_esperada": "Antes da internação e do procedimento cirúrgico cardíaco, é obrigatória a realização de exames pré-operatórios padrão. Isso inclui a coleta de exames de sangue (como hemograma e testes de coagulação), eletrocardiograma (ECG) e exames de imagem adicionais recomendados pela equipe médica para garantir a segurança clínica.",
        "kb1_artigos": "Sim, existem vários exames específicos que precisam ser feitos antes da internação para a cirurgia cardíaca. Alguns dos exames comuns incluem: exames de sangue, eletrocardiograma, radiografia de tórax...",
        "kb2_faq": "Esta é uma questão específica do seu histórico médico...",
        "kb3_combinada": "Sim, existem exames específicos que precisam ser feitos antes da internação...",
    },
    {
        "pergunta": "Há alguma medicação que devo suspender antes da cirurgia?",
        "resposta_esperada": "Determinados medicamentos devem ser obrigatoriasmente suspensos antes da cirurgia cardíaca, especialmente anticoagulantes, antiagregantes plaquetários ou remédios para diabetes, devido ao risco de sangramento ou complicações glicêmicas. No entanto, o tempo exato de suspensão e quais medicamentos interromper devem seguir estritamente a orientação individualizada do seu cirurgião ou cardiologista.",
        "kb1_artigos": "Baseado no contexto fornecido, não há menção específica sobre medicamentos que devem ser suspensos antes da cirurgia...",
        "kb2_faq": "Esta é uma questão específica do seu histórico médico...",
        "kb3_combinada": "Esta é uma questão específica do seu histórico médico...",
    },
    {
        "pergunta": "DEVO INTERNAR ANTES DA MINHA CIRURGIA CARDÍACA?",
        "resposta_esperada": "Sim, é necessário e recomendável realizar a internação hospitalar regulamentar antes da cirurgia cardíaca. Esse período prévio é fundamental para que a equipe de saúde realize o preparo físico, exames de última hora, banhos antissépticos protocolares e a avaliação final de segurança para o procedimento.",
        "kb1_artigos": "Sim, é comum que os pacientes sejam internados antes da cirurgia cardíaca...",
        "kb2_faq": "Esta é uma questão específica do seu histórico médico...",
        "kb3_combinada": "Sim, é recomendável internar-se antes da cirurgia cardíaca...",
    },
    {
        "pergunta": "PRECISO FICAR EM JEJUM ANTES DA INTERNAÇÃO?",
        "resposta_esperada": "Não é necessário iniciar o jejum antes de chegar ao hospital para a internação regular, a menos que haja uma ordem médica específica. O protocolo de jejum absoluto (geralmente de 8 horas) será instituído e controlado pela equipe de enfermagem dentro do hospital, preparando o paciente para o horário programado da cirurgia.",
        "kb1_artigos": "Baseado no contexto fornecido, não há menção específica sobre o jejum antes da internação...",
        "kb2_faq": "Sim, é necessário realizar jejum antes da internação.",
        "kb3_combinada": "Sim, é necessário jejum antes da cirurgia. O paciente foi orientado a realizar um jejum de 8 horas...",
    },
    {
        "pergunta": "QUAIS SÃO OS CUIDADOS COM A HIGIENE ANTES DA CIRURGIA?",
        "resposta_esperada": "Os cuidados com a higiene antes da cirurgia são cruciais para prevenir infecções do sítio cirúrgico. O paciente deve realizar banhos minuciosos utilizando sabonete antisséptico (como clorexidina), lavando bem o cabelo e o corpo, conforme o protocolo institucional. Além disso, a tricotomia (remoção de pelos) da região cirúrgica deve ser feita exclusivamente pela equipe de enfermagem no hospital.",
        "kb1_artigos": "Baseado no contexto fornecido, as orientações de higiene incluem a realização de tricotomia...",
        "kb2_faq": "Baseado no contexto fornecido, o banho pré-operatório com antisséptico deve ser realizado pelo paciente...",
        "kb3_combinada": "Baseado no contexto fornecido, o cuidado com a higiene inclui o banho pré-operatório com antisséptico...",
    },
    {
        "pergunta": "O QUE DEVO LEVAR PARA O HOSPITAL NO DIA DA INTERNAÇÃO?",
        "resposta_esperada": "No dia da internação, o paciente deve levar todos os seus exames de saúde recentes, documentos de identificação pessoal, uma lista atualizada de medicamentos em uso e objetos de higiene pessoal. Roupas confortáveis e fáceis de vestir para o momento da alta também são recomendadas, evitando o excesso de bagagem.",
        "kb1_artigos": "Baseado no contexto fornecido, não há informações sobre o que levar para o hospital no dia da internação...",
        "kb2_faq": "O paciente deve trazer pertences de higiene pessoal e roupas confortáveis...",
        "kb3_combinada": "O paciente deve trazer pertences de higiene pessoal e roupas confortáveis...",
    },
    {
        "pergunta": "Como é o pós-operatório na UTI?",
        "resposta_esperada": "O pós-operatório imediato na UTI envolve monitoramento contínuo e intensivo. O paciente acordará conectado a monitores, recebendo oxigênio e com acessos venosos. A equipe de enfermagem e médica controlará constantemente os sinais vitais, dor e exames laboratoriais. A extubação (retirada do tubo de respiração) costuma ocorrer poucas horas após o término da cirurgia, assim que o paciente apresentar estabilidade.",
        "kb1_artigos": "O pós-operatório na UTI após uma cirurgia cardíaca é um período crítico que requer monitoramento intensivo...",
        "kb2_faq": "Baseado no contexto fornecido, o paciente irá para a UTI ainda dormindo sob efeito anestésico...",
        "kb3_combinada": "O pós-operatório na UTI envolve monitoramento intensivo e contínuo...",
    },
    {
        "pergunta": "Sentir dor após a cirurgia é normal? O que é feito para controlar?",
        "resposta_esperada": "Sim, sentir algum nível de dor ou desconforto na região do tórax e das incisões é considerado normal devido ao trauma cirúrgico. Para controlar e mitigar esse sintoma, a equipe de saúde administra analgésicos potentes programados e doses de resgate diretamente na veia ou por via oral, avaliando constantemente a escala de dor do paciente.",
        "kb1_artigos": "Sim, sentir dor após a cirurgia cardíaca é considerado normal devido ao procedimento invasivo...",
        "kb2_faq": "Baseado no contexto fornecido, o controle da dor é feito por meio de analgésicos...",
        "kb3_combinada": "Sim, sentir dor após a cirurgia cardíaca é normal. O controle é feito por analgésicos...",
    },
    {
        "pergunta": "QUANDO PODEREI RECEBER VISITAS NA UTI?",
        "resposta_esperada": "As visitas na Unidade de Terapia Intensiva (UTI) são restritas e seguem horários fixos determinados pela instituição hospitalar para garantir o descanso do paciente e a rotina de cuidados. A equipe de enfermagem informará os familiares sobre os horários permitidos e o número de visitantes por vez logo após a transferência do paciente para a unidade.",
        "kb1_artigos": "Baseado no contexto fornecido, não há informações sobre os horários de visita na UTI...",
        "kb2_faq": "Os horários de visita na UTI variam de acordo com a instituição hospitalar...",
        "kb3_combinada": "Os horários de visita na UTI variam de acordo com a instituição hospitalar...",
    },
    {
        "pergunta": "Como funciona a transferência da UTI para o quarto?",
        "resposta_esperada": "A transferência da UTI para a enfermaria ou quarto ocorre assim que o paciente atinge estabilidade hemodinâmica completa. Isso significa que ele deve estar respirando bem sem auxílio de aparelhos, com a dor controlada, exames laboratoriais estáveis e sem a necessidade de medicações contínuas para manter a pressão arterial.",
        "kb1_artigos": "A transferência da UTI para o quarto ocorre após o paciente apresentar estabilidade clínica...",
        "kb2_faq": "Baseado no contexto fornecido, a transferência ocorre quando o paciente apresenta estabilidade...",
        "kb3_combinada": "A transferência do paciente da UTI para o quarto ocorre mediante estabilidade clínica...",
    },
    {
        "pergunta": "Quais cuidados devo ter com a alimentação no hospital e em casa?",
        "resposta_esperada": "A alimentação deve ser leve, balanceada e saudável. Tanto no hospital quanto em casa, é fundamental reduzir drasticamente o consumo de sal e gorduras para proteger o sistema cardiovascular. Deve-se priorizar a ingestão de frutas, legumes, verduras, carnes magras e manter uma boa hidratação, seguindo as restrições hídricas específicas caso tenham sido prescritas pelo médico.",
        "kb1_artigos": "Tanto no hospital quanto em casa, cuidados com a alimentação são essenciais para a recuperação...",
        "kb2_faq": "Baseado no contexto fornecido, a dieta deve ser equilibrada, rica em nutrientes e com restrição de sódio...",
        "kb3_combinada": "Os cuidados com a alimentação incluem uma dieta equilibrada, rica em nutrientes e com baixo teor de sódio...",
    },
    {
        "pergunta": "Como e quando é feita a retirada dos pontos e drenos?",
        "resposta_esperada": "A retirada dos drenos cirúrgicos costuma ser feita ainda no hospital, assim que o volume de drenagem diminui de forma segura. Já a retirada dos pontos ou grampos da pele das incisões (esterno e perna) ocorre geralmente entre 10 a 14 dias após a cirurgia, podendo ser realizada no momento da alta ou na primeira consulta de retorno ambulatorial.",
        "kb1_artigos": "A retirada de pontos e drenos após cirurgia cardíaca é realizada pela equipe de saúde...",
        "kb2_faq": "A retirada de drenos ocorre geralmente nos primeiros dias de pós-operatório na UTI ou quarto...",
        "kb3_combinada": "A retirada de drenos e pontos ocorre sob supervisão médica. Os drenos são retirados no hospital...",
    },
    {
        "pergunta": "QUAIS SÃO OS CUIDADOS COM AS CICATRIZES EM CASA?",
        "resposta_esperada": "Para cuidar das cicatrizes em casa, lave a ferida operatória diariamente durante o banho utilizando apenas água morna e sabonete neutro. Seque a região com uma toalha limpa de forma muito suave e delicada, sem esfregar. Mantenha as incisões limpas, secas e totalmente descobertas, evitando o uso de pomadas, cremes ou curativos fechados, a menos que haja ordem médica.",
        "kb1_artigos": "Baseado no contexto fornecido, os cuidados com as cicatrizes incluem mantê-las limpas e secas...",
        "kb2_faq": "Para limpar a ferida use água morna, siga com banho de sabonete neutro, seque as cicatrizes com delicadeza e mantenha-as descobertas.",
        "kb3_combinada": "Os cuidados com as cicatrizes em casa envolvem lavar a ferida com água morna e sabonete neutro...",
    },
    {
        "pergunta": "QUANDO POSSO VOLTAR A FAZER ATIVIDADES FÍSICAS?",
        "resposta_esperada": "O retorno às atividades físicas deve ser gradual e sem exageros. Nos primeiros três meses após a alta, caminhadas leves em superfícies planas são suficientes e recomendadas. Atividades que exijam esforço exagerado da musculatura do tórax, levantamento de pesos ou elevação abrupta dos braços devem ser evitadas para garantir a completa cicatrização do osso esterno.",
        "kb1_artigos": "O retorno às atividades físicas após cirurgia cardíaca deve ser gradual e supervisionado...",
        "kb2_faq": "Realize atividades físicas sem exagero. Pequenas caminhadas são suficientes nos primeiros três meses...",
        "kb3_combinada": "A retomada de atividades físicas deve ser gradual. Pequenas caminhadas são recomendadas nos primeiros 3 meses...",
    },
    {
        "pergunta": "QUANDO PODEREI VOLTAR A TRABALHAR?",
        "resposta_esperada": "O retorno ao trabalho depende diretamente do tipo de atividade profissional exercida pelo paciente e do seu ritmo de recuperação física. Geralmente, o afastamento inicial varia de 30 a 90 dias. Atividades administrativas e leves podem ser retomadas mais cedo, enquanto profissões que exigem esforço físico demandam maior tempo de repouso e aprovação em consulta médica.",
        "kb1_artigos": "O momento ideal para retornar ao trabalho após uma cirurgia cardíaca depende de vários fatores...",
        "kb2_faq": "O retorno ao trabalho varia conforme a ocupação e o estado clínico do paciente...",
        "kb3_combinada": "O retorno ao trabalho depende da recuperação clínica do paciente e da natureza de suas funções...",
    },
    {
        "pergunta": "QUANDO POSSO VOLTAR A DIRIGIR APÓS A ALTA?",
        "notes": "Puxando o texto do seu FAQ/TCC sobre restrição de 30 dias",
        "resposta_esperada": "Espere no mínimo trinta dias após a alta hospitalar para voltar a dirigir. Esse período é crucial para a consolidação inicial do osso esterno. Ao retornar, tome cuidado extra, pois seus reflexos e força estarão diminuídos. Evite dirigir por longos períodos e, caso sinta cansaço, estacione imediatamente em local seguro e aguarde melhorar.",
        "kb1_artigos": "A literatura científica indica que o tempo de recuperação varia. Pacientes submetidos a procedimentos cirúrgicos de grande porte devem evitar esforços físicos...",
        "kb2_faq": "O paciente deve aguardar o período de trinta dias pós-alta hospitalar para voltar a conduzir veículos automotores.",
        "kb3_combinada": "Espere no mínimo trinta dias após a alta para dirigir, mas tome cuidado, seus reflexos e força estarão diminuídos. Evite força exagerada, não dirija por longos períodos...",
    },
    {
        "pergunta": "POSSO BEBER BEBIDAS ALCOÓLICAS OU FUMAR DEPOIS DA CIRURGIA?",
        "resposta_esperada": "É terminantemente proibido fumar após a cirurgia cardíaca, pois o tabaco prejudica a cicatrização e agrava doenças cardiovasculares. Quanto às bebidas alcoólicas, o consumo deve ser evitado ou realizado em quantidades muito reduzidas e com longos intervalos, garantindo que não haja interação prejudicial com as medicações de uso contínuo.",
        "kb1_artigos": "Baseado no contexto fornecido, não é recomendado fumar ou consumir bebidas alcoólicas após a cirurgia...",
        "kb2_faq": "Não fume. Pode ingerir bebidas alcoólicas em pouca quantidade e longos intervalos, não deixe de tomar suas medicações se beber.",
        "kb3_combinada": "Não fume após a cirurgia. O consumo de álcool deve ser evitado ou feito em quantidades mínimas...",
    },
    {
        "pergunta": "POSSO TOMAR CAFÉ?",
        "resposta_esperada": "O café possui uma ação estimulante importante devido à cafeína, o que pode alterar a frequência cardíaca. Por isso, ele deve ser consumido com moderação e em pouca quantidade durante o período de recuperação, evitando excessos que possam desencadear palpitações ou ansiedade.",
        "kb1_artigos": "Baseado no contexto fornecido, não há informações específicas sobre o consumo de café...",
        "kb2_faq": "O café tem uma ação estimulante importante, beba em pouca quantidade.",
        "kb3_combinada": "O café tem uma ação estimulante importante, beba em pouca quantidade.",
    },
    {
        "pergunta": "Como devo gerenciar meus medicamentos em casa? O que faço se esquecer de tomar uma dose?",
        "resposta_esperada": "Em casa, organize seus medicamentos utilizando caixas organizadoras ou tabelas com horários bem definidos. Nunca altere as doses ou suspenda medicações por conta própria. Caso se esqueça de tomar uma dose, tome-a assim que lembrar, a menos que esteja muito próximo do horário da próxima dose. Se isso acontecer, pule a dose esquecida e siga o cronograma normal, nunca dobrando a dose para compensar.",
        "kb1_artigos": "Gerenciar medicamentos em casa após cirurgia cardíaca exige organização e aderência estrita...",
        "kb2_faq": "Esta é uma questão específica do seu histórico médico e deve ser discutida diretamente com seu médico...",
        "kb3_combinada": "Esta é uma questão específica do seu histórico médico e deve ser discutida diretamente com seu médico...",
    },
    {
        "pergunta": "Quais são os sinais de alerta que indicam que devo ir ao hospital imediatamente?",
        "resposta_esperada": "O paciente deve procurar o serviço de emergência imediatamente caso apresente sinais de alerta graves, tais como: dor forte ou aperto no peito que não melhora com repouso, falta de ar súbita, febre acima de 37,8°C, palpitações cardíacas, desmaios, ou caso note vermelhidão, calor, inchaço ou saída de secreção com mau cheiro nas cicatrizes da cirurgia.",
        "kb1_artigos": "Após a alta hospitalar, os pacientes devem monitorar atentamente o surgimento de sinais de alerta...",
        "kb2_faq": "Procure o hospital imediatamente se apresentar dor no peito intensa, falta de ar inexplicável, febre persistente...",
        "kb3_combinada": "Os sinais de alerta que indicam a necessidade de atendimento médico imediato incluem dor torácica severa...",
    },
    {
        "pergunta": "MEU FAMILIAR VAI CONSEGUIR CUIDAR DE MIM EM CASA?",
        "resposta_esperada": "Sim, o seu familiar será perfeitamente capaz de auxiliar nos cuidados em casa. No momento da alta hospitalar, a equipe de enfermagem fornecerá todas as orientações necessárias por escrito e explicará detalhadamente os cuidados com curativos, medicamentos e sinais de alerta, garantindo que o cuidador se sinta seguro para ajudar na rotina.",
        "kb1_artigos": "Baseado no contexto fornecido, não há informações sobre a capacidade de familiares prestarem cuidados...",
        "kb2_faq": "Sim, ele sairá com orientações do hospital, para que possa lhe auxiliar da melhor forma possível.",
        "kb3_combinada": "Sim, o familiar receberá todas as orientações da equipe de enfermagem antes da alta hospitalar...",
    },
    {
        "pergunta": "Como lidar com a ansiedade e mudanças de humor após a cirurgia?",
        "resposta_esperada": "Mudanças de humor, episódios de choro ou ansiedade são reações comuns e esperadas no pós-operatório de cirurgias de grande porte. É importante conversar abertamente com familiares sobre esses sentimentos, manter uma rotina de repouso e caminhadas leves, e buscar apoio psicológico ou da equipe de saúde do hospital caso esses sintomas se tornem intensos ou persistentes.",
        "kb1_artigos": "Ansiedade e flutuações emocionais são frequentes no período de convalescença pós-cirúrgica...",
        "kb2_faq": "Alterações emocionais podem ocorrer. Recomenda-se manter um ambiente tranquilo e apoio familiar...",
        "kb3_combinada": "É comum apresentar episódios de ansiedade após a cirurgia. O suporte familiar e profissional é recomendado...",
    },
    {
        "pergunta": "Quando será minha primeira consulta de retorno após a alta?",
        "resposta_esperada": "A primeira consulta de retorno ambulatorial com o cirurgião cardíaco ou cardiologista clínico ocorre, de maneira geral, entre 7 a 15 dias após a alta hospitalar. A data e o horário exatos dessa consulta são agendados e informados ao paciente pela equipe administrativa do hospital no momento em que ele recebe a alta.",
        "kb1_artigos": "O primeiro retorno médico pós-alta varia de acordo com o protocolo do serviço de cirurgia...",
        "kb2_faq": "A primeira consulta de retorno deve ser realizada conforme o agendamento fornecido na alta hospitalar...",
        "kb3_combinada": "A consulta de retorno ambulatorial é agendada no momento da alta, ocorrendo nos primeiros dias...",
    },
    {
        "pergunta": "Como posso obter apoio contínuo da equipe de saúde após ir para casa?",
        "resposta_esperada": "Após receber a alta, o paciente pode obter apoio contínuo comparecendo assiduamente às consultas agendadas de retorno ambulatorial. Além disso, o hospital disponibiliza os canais de contato telefônico oficiais da unidade de internação e da equipe de enfermagem navegadora para esclarecer dúvidas pontuais sobre receitas ou sintomas ambulatoriais.",
        "kb1_artigos": "O suporte contínuo no pós-operatório domiciliar baseia-se no acompanhamento ambulatorial regular...",
        "kb2_faq": "Para suporte contínuo, utilize os contatos telefônicos fornecidos no cartão de alta ou procure o ambulatório...",
        "kb3_combinada": "O apoio pós-alta é garantido por meio dos telefones de contato fornecidos e das consultas ambulatoriais...",
    }
]

df = pd.DataFrame(dados_experimento)


# Função para calcular similaridade entre os textos
def calcular_cosseno(texto_gerado, texto_referencia):
    vetor_gerado = model.encode(texto_gerado)
    vetor_referencia = model.encode(texto_referencia)

    dot_product = np.dot(vetor_gerado, vetor_referencia)
    norm_gerado = np.linalg.norm(vetor_gerado)
    norm_referencia = np.linalg.norm(vetor_referencia)

    return dot_product / (norm_gerado * norm_referencia)


print("Processando embeddings e calculando similaridades...")
df["Cosseno_KB1"] = df.apply(
    lambda row: calcular_cosseno(row["kb1_artigos"], row["resposta_esperada"]),
    axis=1,
)
df["Cosseno_KB2"] = df.apply(
    lambda row: calcular_cosseno(row["kb2_faq"], row["resposta_esperada"]),
    axis=1,
)
df["Cosseno_KB3"] = df.apply(
    lambda row: calcular_cosseno(row["kb3_combinada"], row["resposta_esperada"]),
    axis=1,
)

print("\n--- RESULTADOS POR PERGUNTA ---")
for idx, row in df.iterrows():
    print(f"\nPergunta {idx+1}: {row['pergunta']}")
    print(f"  > Cosseno KB1 (Artigos): {row['Cosseno_KB1']:.4f}")
    print(f"  > Cosseno KB2 (FAQ):     {row['Cosseno_KB2']:.4f}")
    print(f"  > Cosseno KB3 (Híbrida): {row['Cosseno_KB3']:.4f}")

print("\n=== METRICAS FINAIS PARA A TABELA DO ARTIGO (CBEB) ===")
print(
    f"KB1 (Artigos): Média = {df['Cosseno_KB1'].mean():.2f} ± {df['Cosseno_KB1'].std():.2f}"
)
print(
    f"KB2 (FAQ):     Média = {df['Cosseno_KB2'].mean():.2f} ± {df['Cosseno_KB2'].std():.2f}"
)
print(
    f"KB3 (Híbrida): Média = {df['Cosseno_KB3'].mean():.2f} ± {df['Cosseno_KB3'].std():.2f}"
)

Carregando o modelo all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processando embeddings e calculando similaridades...

--- RESULTADOS POR PERGUNTA ---

Pergunta 1: Quais são as orientações antes da internação para a cirurgia cardíaca? Preciso parar de fumar e beber antes da internação para a cirurgia?
  > Cosseno KB1 (Artigos): 0.7706
  > Cosseno KB2 (FAQ):     0.5661
  > Cosseno KB3 (Híbrida): 0.3971

Pergunta 2: Existe algum exame específico que preciso fazer antes da internação?
  > Cosseno KB1 (Artigos): 0.8262
  > Cosseno KB2 (FAQ):     0.3975
  > Cosseno KB3 (Híbrida): 0.5354

Pergunta 3: Há alguma medicação que devo suspender antes da cirurgia?
  > Cosseno KB1 (Artigos): 0.5307
  > Cosseno KB2 (FAQ):     0.3856
  > Cosseno KB3 (Híbrida): 0.3856

Pergunta 4: DEVO INTERNAR ANTES DA MINHA CIRURGIA CARDÍACA?
  > Cosseno KB1 (Artigos): 0.7740
  > Cosseno KB2 (FAQ):     0.5341
  > Cosseno KB3 (Híbrida): 0.7708

Pergunta 5: PRECISO FICAR EM JEJUM ANTES DA INTERNAÇÃO?
  > Cosseno KB1 (Artigos): 0.5247
  > Cosseno KB2 (FAQ):     0.5897
  > Cosseno KB3